# 3. AI-Assisted Programming

**기본 경로는 자기 AI 도구로 요청하고 결과 파일을 실제로 완성하는 실습입니다.** 아래 대체 fixture는 AI 계정이나 도구 접근이 없을 때만 씁니다. 고정 fixture의 출력은 실제 모델 응답이나 실제 도구 호출 기록이 아닙니다.

Python 문법 구현은 요구하지 않습니다. 제공된 검사 셀은 그대로 실행하고 결과를 읽으면 됩니다.

## 기본 경로 A. IDE에서 실제 프로그램 완성

1. `3_starter.zip`을 풀고 IDE의 File > Open Folder에서 `3_starter`를 여세요.
2. Terminal > New Terminal에서 `python --version` 또는 `python3 --version`으로 사용할 명령을 확인하세요.
3. AI coding chat을 열고 Add Context의 Files, 파일 mention 또는 드래그로 README.md, inspection.csv, checks.py를 첨부하세요.
4. 아래 요청을 보내고 app.py의 변경 제안을 적용하거나 새 파일에 저장하세요.
5. 명령을 실행해 JSON과HTML을 만들고 browser에서 HTML을 여세요.
6. checker를 실행하고 실패하면 실제 오류와 결과 파일을 붙여 수정을 요청하세요.

**첫 요청**

> Read the three attached files. Create app.py with the Python standard library. Follow the JSON and HTML contract in README. Do not change the CSV or checker. Produce report.json and inspection-report.html.

```text
python app.py --input inspection.csv --output report.json --html inspection-report.html
python checks.py
```

동작하는 명령이python3이면 위python을python3로 바꿉니다. AI가 실행했다고 말해도 실제 출력과 파일을 확인하세요.

## 오류 피드백과 diff

> The unchanged checker fails. The actual command output and result files are attached. Fix app.py only, then run the same generator and checker again.

변경 파일의 diff를 열어 source나checker가 바뀌지 않았는지 확인하세요. 기대값을 고쳐 통과시키는 것은 이 과제의 수정이 아닙니다.

## checkpoint와 작은 확장

첫 통과 뒤 Git 또는 IDE checkpoint로 기준 상태를 보존하세요.

> Add the title “Inspection Review” to the HTML. Keep report.json unchanged. Regenerate both outputs and run checks.py.

변경을 확인한 뒤 최신app.py수정만 Undo 또는 checkpoint로 복원하고 보고서를 다시 생성하세요.

```text
git diff -- app.py
git restore app.py
python app.py --input inspection.csv --output report.json --html inspection-report.html
python checks.py
```

Git명령은 기준commit이 준비된 자기실습폴더에서만 씁니다. 준비되지 않았다면 IDE의복원기능이나자기폴더사본을사용하세요.

## 기본 경로의 완료물

app.py,report.json,inspection-report.html과첫요청·후속요청·실제명령출력·diff·복원기록을남기세요. Python문법을직접구현하는것이아니라AI와파일작업을끝내는것이과제입니다.

## 대체 경로 B. 고정 reference와독립검사

다음셀은AI가만든프로그램이아닙니다. 제공CSV에서정답형식의reference를계산하고임시폴더에서checker를시험합니다. 실제IDE와AI작업을한것으로기록하지마세요.

In [1]:
from pathlib import Path
import tempfile,importlib.util,json,html,shutil
starter=next((p for p in [Path("labs/3_starter"),Path("3_starter")]if p.exists()),None)
if starter is None:raise FileNotFoundError("Unzip 3_starter.zip beside this notebook, or run from the course root")
spec=importlib.util.spec_from_file_location("fixture_checks",starter/"checks.py")
checker=importlib.util.module_from_spec(spec);spec.loader.exec_module(checker)
reference=checker.expected(starter/"inspection.csv")
assert reference["summary"]=={"raw":8,"valid":6,"flagged":4,"missing":1,"invalid":1}
print("Source-derived reference:",reference["summary"])


Source-derived reference: {'raw': 8, 'valid': 6, 'flagged': 4, 'missing': 1, 'invalid': 1}


In [2]:
with tempfile.TemporaryDirectory(prefix="ax-report-check-")as td:
    target=Path(td)
    shutil.copyfile(starter/"inspection.csv",target/"inspection.csv")
    (target/"report.json").write_text(json.dumps(reference),encoding="utf-8")
    head="<h1>Human-authored reference fixture</h1>"+html.escape(str(reference["summary"]))
    rows="<table><tr><th>ID</th><th>Raw score</th><th>Status</th></tr>"
    for row in reference["rows"]:
        rows+="<tr>"+"".join("<td>"+html.escape(str(row[k]))+"</td>"for k in ["id","raw_score","status"])+"</tr>"
    (target/"inspection-report.html").write_text(head+rows+"</table>",encoding="utf-8")
    print("Reference fixture checker:",checker.check(target))
    # Meaningful negative case: dropping a duplicate row must fail.
    broken=dict(reference);broken["rows"]=reference["rows"][:-1]
    (target/"report.json").write_text(json.dumps(broken),encoding="utf-8")
    try:checker.check(target)
    except AssertionError:print("A dropped-row candidate is rejected")
    else:raise AssertionError("Checker accepted a dropped row")


Reference fixture checker: {'raw': 8, 'valid': 6, 'flagged': 4, 'missing': 1, 'invalid': 1}
A dropped-row candidate is rejected


## 대체 경로의과제와해답

- JSON에서행하나를누락하면checker가거부하는지확인하세요.
- missing을score0으로바꿔넣는candidate가왜계약과다른지설명하세요.

첫문제는위셀에서실제로실행했습니다. 아래셀은missing원본이그대로남는것을확인합니다.

In [3]:
missing=[r for r in reference["rows"]if r["status"]=="missing"]
assert len(missing)==1 and missing[0]["raw_score"]=="" and missing[0]["flagged"]is None
print("Missing row remains unknown:",missing[0])

Missing row remains unknown: {'row': 4, 'id': 'A04', 'raw_score': '', 'status': 'missing', 'flagged': None}


## 현재공식사용흐름

[파일문맥첨부](https://code.visualstudio.com/docs/chat/copilot-chat-context), [변경검토와복원](https://code.visualstudio.com/docs/agents/run/review-code-edits). UI와지원기능은사용하는client와harness에따라달라집니다.